# Grafik Data Politikus Indonesia
## Knowledge Graph + LLM Pipeline (Text-to-Cypher | Graph Builder | RAG)

**Stack:** Neo4j AuraDB | OpenRouter API (free models) | Python 3.10
**Dataset:** 52.295 Politikus | 156 Partai | 881 Pendidikan


---
### Arsitektur Sistem
```

```


## Cell 0 - Instalasi Dependencies

In [ ]:
# Install semua dependencies yang diperlukan
!pip install neo4j requests pandas tqdm --quiet

print("Semua dependencies terinstall")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 5.8 MB/s eta 0:00:00
Semua dependencies terinstall


## Cell 1 - Konfigurasi Koneksi

> **Isi credential Neo4j AuraDB dan OpenRouter API key sebelum run.**


In [ ]:
import os
from google.colab import userdata

# Neo4j AuraDB Credentials
NEO4J_URI      = userdata.get('NEO4J_URI')
NEO4J_USER     = userdata.get('NEO4J_USER')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

# OpenRouter API
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

# LLM Model

FREE_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

print("Konfigurasi siap")
print(f"   Model LLM : {FREE_MODEL}")
print(f"   Neo4j URI : {NEO4J_URI[:30]}...")


Konfigurasi siap
   Model LLM : nvidia/nemotron-3-ultra-550b-a55b:free
   Neo4j URI : bolt://54.205.116.77...


## Cell 2 - Koneksi Neo4j & Helper Functions

In [ ]:
from neo4j import GraphDatabase
import requests, json, time

# Neo4j Driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(cypher, params=None):
    """Jalankan Cypher query dan return list of dicts."""
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return [dict(r) for r in result]

# Test koneksi
try:
    info = run_query("RETURN 'Koneksi berhasil!' AS status, datetime() AS waktu")
    print("OK:", info[0]["status"])
    print("   Server time:", info[0]["waktu"])
except Exception as e:
    print("Koneksi gagal:", e)
    print("   Pastikan URI, user, dan password AuraDB sudah benar")

# OpenRouter LLM Helper
def call_llm(system_prompt, user_prompt, temperature=0.1):
    """Panggil LLM via OpenRouter API."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "Graf-Politikus-Indonesia"
    }
    payload = {
        "model": FREE_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": 1024
    }
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers, json=payload, timeout=60
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

# Test LLM
try:
    test = call_llm("You are a helpful assistant.", "Jawab singkat: Ibu kota Indonesia?")
    print("\nLLM aktif:", test[:120])
except Exception as e:
    print("LLM error:", e)


OK: Koneksi berhasil!
   Server time: 2026-06-22T08:02:57.198000000+00:00

LLM aktif: Jakarta.


## Cell 3 - Import Dataset ke Neo4j AuraDB

Data diambil langsung dari GitHub. Proses ini memuat:
- **Node:** Politikus, Partai, Pendidikan, Person
- **Edge:** ANGGOTA_PARTAI, ALUMNI, KERABAT

> Estimasi waktu: 10-20 menit (52K+ nodes via AuraDB free tier)


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/ibrahimamar07/Graf-Data-Politikus-Indonesia/main/dataset_load_to_neo4j"

def import_node(label, csv_name, set_clause, description):
    print(f"Importing {description}...", end=" ", flush=True)
    t0 = time.time()
    try:
        url = f"{BASE_URL}/{csv_name}"
        cypher = (
            f"LOAD CSV WITH HEADERS FROM '{url}' AS row "
            "WITH row WHERE row.`kodeId:ID` IS NOT NULL AND row.`kodeId:ID` <> '' "
            f"MERGE (p:{label} {{kodeId: row.`kodeId:ID`}}) "
            f"SET {set_clause}"
        )
        run_query(cypher)
        count = run_query(f"MATCH (n:{label}) RETURN count(n) AS c")[0]["c"]
        print(f"OK {count:,} nodes ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"ERROR: {e}")

def import_edge(label, csv_name, match_a, match_b, merge_rel, description):
    print(f"Importing {description}...", end=" ", flush=True)
    t0 = time.time()
    try:
        url = f"{BASE_URL}/{csv_name}"
        cypher = (
            f"LOAD CSV WITH HEADERS FROM '{url}' AS row "
            "WITH row WHERE row.`:START_ID` IS NOT NULL AND row.`:END_ID` IS NOT NULL "
            "AND row.`:START_ID` <> '' AND row.`:END_ID` <> '' "
            f"{match_a} "
            f"{match_b} "
            f"{merge_rel}"
        )
        run_query(cypher)
        count = run_query(f"MATCH ()-[r:{label}]->() RETURN count(r) AS c")[0]["c"]
        print(f"OK {count:,} relasi ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"ERROR: {e}")

# STEP 1: Constraints
print("Membuat constraints...")
for c in [
    "CREATE CONSTRAINT pol_id IF NOT EXISTS FOR (p:Politikus)  REQUIRE p.kodeId IS UNIQUE",
    "CREATE CONSTRAINT par_id IF NOT EXISTS FOR (p:Partai)     REQUIRE p.kodeId IS UNIQUE",
    "CREATE CONSTRAINT pen_id IF NOT EXISTS FOR (p:Pendidikan) REQUIRE p.kodeId IS UNIQUE",
    "CREATE CONSTRAINT per_id IF NOT EXISTS FOR (p:Person)     REQUIRE p.kodeId IS UNIQUE",
]:
    try:
        run_query(c)
    except:
        pass
print("Constraints siap")

# STEP 2: Import Nodes
import_node("Politikus",  "nodes_politikus.csv",  "p.nama = row.nama", "Node Politikus")
import_node("Partai",     "nodes_partai.csv",     "p.nama = row.nama",                     "Node Partai")
import_node("Pendidikan", "nodes_pendidikan.csv", "p.nama = row.nama",                     "Node Pendidikan")
import_node("Person",     "nodes_person.csv",     "p.nama = row.nama",                     "Node Person")

# STEP 3: Import Edges
import_edge(
    "ANGGOTA_PARTAI", "edges_anggota_partai.csv",
    "MATCH (a:Politikus {kodeId: row.`:START_ID`})",
    "MATCH (b:Partai    {kodeId: row.`:END_ID`})",
    "MERGE (a)-[:ANGGOTA_PARTAI]->(b)",
    "Edge ANGGOTA_PARTAI"
)
import_edge(
    "ALUMNI", "edges_alumni.csv",
    "MATCH (a:Politikus  {kodeId: row.`:START_ID`})",
    "MATCH (b:Pendidikan {kodeId: row.`:END_ID`})",
    "MERGE (a)-[:ALUMNI]->(b)",
    "Edge ALUMNI"
)
import_edge(
    "KERABAT", "edges_kerabat_politikus.csv",
    "MATCH (a:Politikus {kodeId: row.`:START_ID`})",
    "MATCH (b:Politikus {kodeId: row.`:END_ID`})",
    "MERGE (a)-[:KERABAT {tipe: row.tipe}]->(b)",
    "Edge KERABAT Politikus-Politikus"
)
import_edge(
    "KERABAT", "edges_kerabat_person.csv",
    "MATCH (a:Politikus {kodeId: row.`:START_ID`})",
    "MATCH (b:Person    {kodeId: row.`:END_ID`})",
    "MERGE (a)-[:KERABAT {tipe: row.tipe}]->(b)",
    "Edge KERABAT Politikus-Person"
)

# Verifikasi
print("\n" + "="*50)
print("RINGKASAN DATABASE")
print("="*50)
for row in run_query("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS jumlah ORDER BY jumlah DESC"):
    print(f"   {row['label']:15s}: {row['jumlah']:>8,} nodes")
print()
for row in run_query("MATCH ()-[r]->() RETURN type(r) AS relasi, count(r) AS jumlah"):
    print(f"   {row['relasi']:20s}: {row['jumlah']:>8,} relasi")


Membuat constraints...
Constraints siap
Importing Node Politikus... OK 52,295 nodes (2.6s)
Importing Node Partai... OK 156 nodes (0.3s)
Importing Node Pendidikan... OK 881 nodes (0.3s)
Importing Node Person... OK 1,024 nodes (0.3s)
Importing Edge ANGGOTA_PARTAI... OK 50,485 relasi (2.2s)
Importing Edge ALUMNI... OK 2,875 relasi (0.5s)
Importing Edge KERABAT Politikus-Politikus... OK 549 relasi (0.4s)
Importing Edge KERABAT Politikus-Person... OK 1,746 relasi (0.6s)

RINGKASAN DATABASE
   Politikus      :   52,295 nodes
   Person         :    1,024 nodes
   Pendidikan     :      881 nodes
   Partai         :      156 nodes

   ANGGOTA_PARTAI      :   50,485 relasi
   ALUMNI              :    2,875 relasi
   KERABAT             :    1,746 relasi


## Cell 4 - Graph Analytics (GDS)

Menjalankan algoritma:
- **PageRank** - tokoh paling berpengaruh
- **Betweenness Centrality** - jembatan jaringan
- **Louvain Community Detection** - pengelompokan komunitas

> Memerlukan Neo4j GDS Plugin (sudah termasuk di AuraDB).


In [ ]:
# Cek apakah GDS tersedia
try:
    gds_check = run_query("RETURN gds.version() AS ver")
    print(f"GDS Plugin aktif: v{gds_check[0]['ver']}")
    GDS_AVAILABLE = True
except Exception as e:
    print(f"GDS tidak tersedia: {e}")
    print("Melanjutkan dengan fallback Cypher analytics...")
    GDS_AVAILABLE = False

if GDS_AVAILABLE:
    # Drop graph projection jika sudah ada
    try:
        run_query("CALL gds.graph.drop('graf_politik', false)")
    except:
        pass

    # Buat Graph Projection
    print("Membuat graph projection...", end=" ")
    proj = run_query(
        "CALL gds.graph.project("
        "  'graf_politik',"
        "  ['Politikus', 'Partai', 'Pendidikan', 'Person'],"
        "  {"
        "    ANGGOTA_PARTAI: {orientation: 'UNDIRECTED'},"
        "    ALUMNI:         {orientation: 'UNDIRECTED'},"
        "    KERABAT:        {orientation: 'UNDIRECTED'}"
        "  }"
        ") YIELD graphName, nodeCount, relationshipCount"
    )
    print(f"OK {proj[0]['nodeCount']:,} nodes, {proj[0]['relationshipCount']:,} relasi")

    # PageRank
    print("Menghitung PageRank...", end=" ")
    pr = run_query(
        "CALL gds.pageRank.write('graf_politik', {"
        "  maxIterations: 20, dampingFactor: 0.85,"
        "  writeProperty: 'pagerank'"
        "}) YIELD ranIterations, didConverge"
    )
    print(f"OK {pr[0]['ranIterations']} iterasi, converge={pr[0]['didConverge']}")

    # Betweenness Centrality
    print("Menghitung Betweenness Centrality...", end=" ")
    bc = run_query(
        "CALL gds.betweenness.write('graf_politik', {"
        "  writeProperty: 'betweenness'"
        "}) YIELD centralityDistribution, nodePropertiesWritten"
    )
    print(f"OK {bc[0]['nodePropertiesWritten']:,} nodes")

    # Louvain
    print("Menjalankan Louvain Community Detection...", end=" ")
    lv = run_query(
        "CALL gds.louvain.write('graf_politik', {"
        "  writeProperty: 'community'"
        "}) YIELD communityCount, modularity"
    )
    print(f"OK {lv[0]['communityCount']} komunitas, modularity={lv[0]['modularity']:.4f}")

    # Tampilkan hasil
    print("\n" + "="*60)
    print("TOP 10 POLITIKUS PALING BERPENGARUH (PageRank)")
    print("="*60)
    top_pr = run_query(
        "MATCH (p:Politikus) WHERE p.pagerank IS NOT NULL AND p.nama <> '' "
        "RETURN p.nama AS nama, round(p.pagerank, 4) AS pagerank, p.community AS komunitas "
        "ORDER BY pagerank DESC LIMIT 10"
    )
    for i, r in enumerate(top_pr, 1):
        print(f"  {i:2}. {r['nama']:<35} PR={r['pagerank']:.4f}  Komunitas={r['komunitas']}")

    print("\n" + "="*60)
    print("TOP 10 DINASTI POLITIK (KERABAT antar Politikus)")
    print("="*60)
    dinasti = run_query(
        "MATCH (p:Politikus)-[:KERABAT]->(k:Politikus) WHERE p.nama <> '' "
        "RETURN p.nama AS politikus, count(k) AS jumlah_kerabat "
        "ORDER BY jumlah_kerabat DESC LIMIT 10"
    )
    for i, r in enumerate(dinasti, 1):
        print(f"  {i:2}. {r['politikus']:<35} {r['jumlah_kerabat']} kerabat politikus")

    print("\n" + "="*60)
    print("TOP 10 UNIVERSITAS PENCETAK POLITIKUS")
    print("="*60)
    univ = run_query(
        "MATCH (p:Politikus)-[:ALUMNI]->(u:Pendidikan) WHERE u.nama IS NOT NULL AND u.nama <> '' "
        "RETURN u.nama AS universitas, count(p) AS alumni ORDER BY alumni DESC LIMIT 10"
    )
    for i, r in enumerate(univ, 1):
        print(f"  {i:2}. {r['universitas'][:45]:<45} {r['alumni']:>5,} alumni")

else:
    print("\nAnalytics fallback - degree centrality via Cypher")
    top_deg = run_query(
        "MATCH (p:Politikus) "
        "WITH p, size([(p)-[]-() | 1]) AS degree "
        "WHERE p.nama <> '' "
        "RETURN p.nama AS nama, degree ORDER BY degree DESC LIMIT 10"
    )
    for i, r in enumerate(top_deg, 1):
        print(f"  {i:2}. {r['nama']:<35} degree={r['degree']}")


GDS Plugin aktif: v2026.04.0
Membuat graph projection... OK 54,356 nodes, 110,212 relasi
Menghitung PageRank... OK 20 iterasi, converge=False
Menghitung Betweenness Centrality... OK 54,356 nodes
Menjalankan Louvain Community Detection... OK 3269 komunitas, modularity=0.8961

TOP 10 POLITIKUS PALING BERPENGARUH (PageRank)
   1. Zakaria bin Muhammad Amin           PR=11.1148  Komunitas=52583
   2. Sukarno                             PR=9.1302  Komunitas=52435
   3. Basuki Tjahaja Purnama              PR=6.4614  Komunitas=48042
   4. Megawati Soekarnoputri              PR=5.9244  Komunitas=52435
   5. Thariq Halilintar                   PR=5.8750  Komunitas=50991
   6. Sutan Takdir Alisjahbana            PR=5.5667  Komunitas=52583
   7. Anies Baswedan                      PR=5.3386  Komunitas=52435
   8. Hary Tanoesoedibjo                  PR=5.2919  Komunitas=52409
   9. Muhammad Jusuf Kalla The Great Devilling PR=5.0664  Komunitas=53165
  10. Sutan Sjahrir                       PR=4.970

## Cell 5 - Komponen 1: Text-to-Cypher (LLM)

LLM menerjemahkan pertanyaan bahasa alami ke query Cypher yang valid untuk Neo4j.

**Schema Graf:**
- Nodes: `Politikus {kodeId, nama, pagerank, betweenness, community}`, `Partai {kodeId, nama}`, `Pendidikan {kodeId, nama}`, `Person {kodeId, nama}`
- Relasi: `ANGGOTA_PARTAI`, `ALUMNI`, `KERABAT {tipe}`


In [ ]:
import json, re

# Schema untuk Text-to-Cypher
GRAPH_SCHEMA = (
    "Neo4j Graph Schema - Graf Politikus Indonesia:\n\n"
    "NODE LABELS & PROPERTIES:\n"
    "- Politikus {kodeId: string, nama: string, pagerank: float, betweenness: float, community: int}\n"
    "- Partai    {kodeId: string, nama: string, pagerank: float, betweenness: float}\n"
    "- Pendidikan{kodeId: string, nama: string}\n"
    "- Person    {kodeId: string, nama: string}\n\n"
    "RELATIONSHIP TYPES:\n"
    "- (Politikus)-[:ANGGOTA_PARTAI]->(Partai)\n"
    "- (Politikus)-[:ALUMNI]->(Pendidikan)\n"
    "- (Politikus)-[:KERABAT {tipe: string}]->(Politikus)\n"
    "- (Politikus)-[:KERABAT {tipe: string}]->(Person)\n\n"
    "RULES:\n"
    "- Gunakan CONTAINS dan toLower() untuk pencarian nama\n"
    "- Selalu tambahkan LIMIT (max 20)\n"
    "- Untuk ranking gunakan pagerank atau betweenness"
)

TEXT_TO_CYPHER_SYSTEM = (
    "Anda adalah expert Neo4j Cypher query generator untuk graf politikus Indonesia.\n\n"
    + GRAPH_SCHEMA +
    "\n\nTUGAS: Terjemahkan pertanyaan natural language ke Cypher query yang valid.\n\n"
    "OUTPUT FORMAT:\n"
    "- Hanya output Cypher query saja, tanpa penjelasan\n"
    "- Tidak perlu markdown code block\n"
    "- Query harus langsung bisa dijalankan di Neo4j\n"
    "- Selalu tambahkan LIMIT untuk mencegah hasil terlalu banyak\n\n"
    "CONTOH:\n"
    "Q: Siapa politikus paling berpengaruh?\n"
    "A: MATCH (p:Politikus) WHERE p.pagerank IS NOT NULL AND p.nama <> '' "
    "RETURN p.nama AS nama, p.pagerank AS skor ORDER BY skor DESC LIMIT 10\n\n"
    "Q: Universitas apa yang paling banyak mencetak politikus?\n"
    "A: MATCH (p:Politikus)-[:ALUMNI]->(u:Pendidikan) WHERE u.nama IS NOT NULL "
    "RETURN u.nama AS universitas, count(p) AS jumlah ORDER BY jumlah DESC LIMIT 10"
)

def text_to_cypher(question):
    """Terjemahkan pertanyaan NL ke Cypher query menggunakan LLM."""
    try:
        cypher_raw = call_llm(TEXT_TO_CYPHER_SYSTEM, f"Pertanyaan: {question}")
    except KeyError as e:
        print(f"ERROR LLM: Respon OpenRouter API tidak memiliki kunci 'choices'. Ini mungkin indikasi API key salah, rate limit, atau masalah internal LLM. Pertanyaan: '{question}'. Error: {e}")
        return None # Indicate failure to generate Cypher
    except Exception as e:
        print(f"ERROR LLM: Terjadi kesalahan tak terduga saat memanggil OpenRouter API untuk pertanyaan: '{question}'. Error: {e}")
        return None

    cypher = cypher_raw.strip()
    for fence in ["```cypher", "```", "cypher"]:
        cypher = cypher.replace(fence, "").strip()
    return cypher

def ask_graph(question, verbose=True):
    """Pipeline: NL Question -> Cypher -> Neo4j -> Results."""
    if verbose:
        print(f"\nPertanyaan: {question}")
        print("-" * 60)

    cypher = text_to_cypher(question)
    if cypher is None: # Check if text_to_cypher failed to generate Cypher
        if verbose:
            print("Gagal membuat Cypher query dari pertanyaan.")
        return []

    if verbose:
        print(f"Cypher: {cypher}")

    try:
        results = run_query(cypher)
        if verbose:
            print(f"Hasil ({len(results)} rows):")
            for i, r in enumerate(results[:10], 1):
                print(f"   {i:2}. {r}")
        return results
    except Exception as e:
        if verbose:
            print(f"Query error: {e}")
            print("Mencoba perbaikan...")
        retry_prompt = f"Query ini error: {cypher}\nError: {e}\nPerbaiki query saja."
        cypher2 = text_to_cypher(retry_prompt) # Use text_to_cypher again for retry
        if cypher2 is None: # Check if retry also failed
            if verbose:
                print("Gagal memperbaiki Cypher query.")
            return []

        try:
            results = run_query(cypher2)
            if verbose:
                print(f"Retry berhasil ({len(results)} rows):")
                for i, r in enumerate(results[:10], 1):
                    print(f"   {i:2}. {r}")
            return results
        except Exception as e2:
            if verbose:
                print(f"Retry gagal: {e2}")
            return []

# Demo Text-to-Cypher
print("=" * 60)
print("DEMO TEXT-TO-CYPHER")
print("=" * 60)

demo_questions = [
    "Siapa 5 politikus dengan pagerank tertinggi?",
    "Partai apa yang memiliki anggota terbanyak?",
    "Politikus mana yang punya kerabat politikus paling banyak (dinasti)?",
    "Universitas mana yang paling banyak menghasilkan politikus?",
]

for q in demo_questions:
    ask_graph(q)
    print()

DEMO TEXT-TO-CYPHER

Pertanyaan: Siapa 5 politikus dengan pagerank tertinggi?
------------------------------------------------------------
ERROR LLM: Respon OpenRouter API tidak memiliki kunci 'choices'. Ini mungkin indikasi API key salah, rate limit, atau masalah internal LLM. Pertanyaan: 'Siapa 5 politikus dengan pagerank tertinggi?'. Error: 'choices'
Gagal membuat Cypher query dari pertanyaan.


Pertanyaan: Partai apa yang memiliki anggota terbanyak?
------------------------------------------------------------
Cypher: MATCH (p:Politikus)-[:ANGGOTA_PARTAI]->(partai:Partai) WHERE partai.nama IS NOT NULL RETURN partai.nama AS partai, count(p) AS jumlah_anggota ORDER BY jumlah_anggota DESC LIMIT 10
Hasil (10 rows):
    1. {'partai': 'Partai Golongan Karya', 'jumlah_anggota': 3578}
    2. {'partai': 'Partai Demokrat (Indonesia)', 'jumlah_anggota': 3438}
    3. {'partai': 'Partai Demokrasi Indonesia Perjuangan', 'jumlah_anggota': 3437}
    4. {'partai': 'Partai Amanat Nasional', 'jumlah_a

## Cell 6 - Komponen 2: Graph Builder (LLM)

LLM mengekstrak entitas & relasi dari teks berita politik tidak terstruktur,
lalu langsung mempopulasi node dan edge baru ke Neo4j.

**Input:** Teks berita politik Indonesia
**Output:** Node & relasi baru di Neo4j dengan label `:BeritaEntity`


In [ ]:
import json, re

GRAPH_BUILDER_SYSTEM = (
    """Anda adalah sistem ekstraksi informasi untuk knowledge graph politikus Indonesia.

Dari teks yang diberikan, ekstrak entitas dan relasi politik dalam format JSON.

OUTPUT FORMAT (JSON saja, tanpa penjelasan lain):
{
  "entities": [
    {"id": "unik_id", "tipe": "Politikus|Partai|Jabatan|Kebijakan|Lokasi", "nama": "nama entitas", "atribut": {}}
  ],
  "relasi": [
    {"dari": "unik_id_1", "tipe_relasi": "MENJABAT|ANGGOTA|MENDUKUNG|MENOLAK|TERKAIT", "ke": "unik_id_2", "atribut": {}}
  ]
}

ATURAN:
- id harus snake_case unik
- Fokus pada entitas politik yang nyata

CONTOH OUTPUT JSON:
{
  "entities": [
    {"id": "joko_widodo", "tipe": "Politikus", "nama": "Joko Widodo"},
    {"id": "mahfud_md", "tipe": "Politikus", "nama": "Mahfud MD"},
    {"id": "pdip", "tipe": "Partai", "nama": "PDIP"}
  ],
  "relasi": [
    {"dari": "mahfud_md", "tipe_relasi": "CALON_WAPRES", "ke": "ganjar_pranowo"},
    {"dari": "ganjar_pranowo", "tipe_relasi": "ANGGOTA", "ke": "pdip"}
  ]
}

ATURAN:
- id harus snake_case unik
- Fokus pada entitas politik yang nyata
- Tipe relasi huruf kapital dengan underscore
- Output JSON valid saja"""
)

def extract_entities_from_text(text):
    """Ekstrak entitas dan relasi dari teks menggunakan LLM."""
    raw = ""
    try:
        raw = call_llm(GRAPH_BUILDER_SYSTEM, f"Teks berita:\n\n{text}", temperature=0.0)
    except KeyError as e:
        print(f"ERROR LLM: Respon OpenRouter API tidak memiliki kunci 'choices' saat ekstraksi entitas. Ini mungkin indikasi API key salah, rate limit, atau masalah internal LLM. Teks: '{text[:50]}...'. Error: {e}")
        return {"entities": [], "relasi": []} # Return empty to prevent crash
    except Exception as e:
        print(f"ERROR LLM: Terjadi kesalahan tak terduga saat memanggil OpenRouter API untuk ekstraksi entitas dari teks: '{text[:50]}...'. Error: {e}")
        return {"entities": [], "relasi": []}

    print(f"Raw LLM Response: {raw[:500]}...") # Print raw response for debugging

    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError as jde:
            print(f"JSON Parsing Error (regex match): {jde}")
            print(f"Attempted JSON: {json_match.group()[:500]}...")
            pass
        except Exception as ex:
            print(f"Unexpected Parsing Error (regex match): {ex}")
            pass
    try:
        return json.loads(raw)
    except json.JSONDecodeError as jde:
        print(f"JSON Parsing Error (full raw): {jde}")
        print(f"Attempted JSON: {raw[:500]}...")
        pass
    except Exception as ex:
        print(f"Unexpected Parsing Error (full raw): {ex}")
        pass

    print("WARNING: Gagal memparsing JSON dari respons LLM. Mengembalikan struktur kosong.")
    return {"entities": [], "relasi": []}

def populate_graph_from_extraction(extracted, source_text=""):
    """Masukkan entitas & relasi ke Neo4j."""
    stats = {"nodes_created": 0, "rels_created": 0, "errors": []}

    for entity in extracted.get("entities", []):
        tipe = entity.get("tipe", "Entity")
        nama = entity.get("nama", "")
        eid  = entity.get("id", "")
        if not nama or not eid:
            continue
        try:
            safe_label = re.sub(r'[^A-Za-z]', '', tipe) or "Entity"
            run_query(
                f"MERGE (e:BeritaEntity:{safe_label} {{extractedId: $eid}}) "
                "SET e.nama = $nama, e.tipe = $tipe, e.source = $src",
                {"eid": eid, "nama": nama, "tipe": tipe, "src": source_text[:100]}
            )
            stats["nodes_created"] += 1
        except Exception as ex:
            stats["errors"].append(str(ex))

    for rel in extracted.get("relasi", []):
        dari = rel.get("dari", "")
        ke   = rel.get("ke", "")
        tipe_rel = re.sub(r'[^A-Z_]', '', rel.get("tipe_relasi", "TERKAIT").upper()) or "TERKAIT"
        if not dari or not ke:
            continue
        try:
            run_query(
                f"MATCH (a:BeritaEntity {{extractedId: $dari}}) "
                "MATCH (b:BeritaEntity {extractedId: $ke}) "
                f"MERGE (a)-[r:{tipe_rel}]->(b)",
                {"dari": dari, "ke": ke}
            )
            stats["rels_created"] += 1
        except Exception as ex:
            stats["errors"].append(str(ex))

    return stats

def link_to_existing_graph(extracted):
    """Link BeritaEntity ke node Politikus/Partai yang sudah ada."""
    linked = 0
    for entity in extracted.get("entities", []):
        nama = entity.get("nama", "")
        tipe = entity.get("tipe", "")
        eid  = entity.get("id", "")
        if not nama or not eid:
            continue
        try:
            if tipe == "Politikus":
                run_query(
                    "MATCH (be:BeritaEntity {extractedId: $eid}) "
                    "MATCH (p:Politikus) WHERE toLower(p.nama) CONTAINS toLower($nama) "
                    "MERGE (be)-[:REFERS_TO]->(p)",
                    {"eid": eid, "nama": nama}
                )
            elif tipe == "Partai":
                run_query(
                    "MATCH (be:BeritaEntity {extractedId: $eid}) "
                    "MATCH (p:Partai) WHERE toLower(p.nama) CONTAINS toLower($nama) "
                    "MERGE (be)-[:REFERS_TO]->(p)",
                    {"eid": eid, "nama": nama}
                )
            linked += 1
        except:
            pass
    return linked

# Demo Graph Builder
SAMPLE_BERITA = [
    (
        "Presiden Joko Widodo mengumumkan reshuffle kabinet hari ini. "
        "Menteri Koordinator Bidang Politik Hukum dan Keamanan Mahfud MD mengundurkan diri "
        "untuk fokus pada pencalonannya sebagai calon wakil presiden mendampingi Ganjar Pranowo dari PDIP. "
        "Posisi Mahfud MD akan diisi oleh Hadi Tjahjanto, mantan Panglima TNI."
    ),
    (
        "Partai Golkar menetapkan Airlangga Hartarto sebagai ketua umum untuk periode berikutnya. "
        "Keputusan ini diambil dalam Musyawarah Nasional Luar Biasa yang dihadiri perwakilan dari 34 provinsi. "
        "Prabowo Subianto dari Gerindra menyatakan dukungannya atas keputusan Golkar tersebut."
    )
]

print("=" * 60)
print("DEMO GRAPH BUILDER (LLM Entity Extraction)")
print("=" * 60)

total_nodes = 0
total_rels  = 0

for i, berita in enumerate(SAMPLE_BERITA, 1):
    print(f"\nBerita {i}:")
    print(f"   {berita[:100]}...")

    print("Mengekstrak entitas dengan LLM...")
    extracted = extract_entities_from_text(berita)

    print(f"Entitas ditemukan ({len(extracted.get('entities', []))}):")
    for e in extracted.get("entities", []):
        print(f"   [{e.get('tipe','?'):12s}] {e.get('nama','?')}")

    print(f"Relasi ditemukan ({len(extracted.get('relasi', []))}):")
    for r in extracted.get("relasi", []):
        print(f"   {r.get('dari','?'):20s} --[{r.get('tipe_relasi','?')}]--> {r.get('ke','?')}")

    stats = populate_graph_from_extraction(extracted, berita[:100])
    total_nodes += stats["nodes_created"]
    total_rels  += stats["rels_created"]
    print(f"Ditambahkan ke Neo4j: {stats['nodes_created']} nodes, {stats['rels_created']} relasi")

    linked = link_to_existing_graph(extracted)
    print(f"Dihubungkan ke graph utama: {linked} entitas")

print(f"\nTOTAL Graph Builder: {total_nodes} nodes baru, {total_rels} relasi baru")
be_count = run_query("MATCH (n:BeritaEntity) RETURN count(n) AS c")[0]["c"]
print(f"Total BeritaEntity di Neo4j: {be_count}")

DEMO GRAPH BUILDER (LLM Entity Extraction)

Berita 1:
   Presiden Joko Widodo mengumumkan reshuffle kabinet hari ini. Menteri Koordinator Bidang Politik Huku...
Mengekstrak entitas dengan LLM...
Raw LLM Response: {
  "entities": [
    {"id": "joko_widodo", "tipe": "Politikus", "nama": "Joko Widodo"},
    {"id": "mahfud_md", "tipe": "Politikus", "nama": "Mahfud MD"},
    {"id": "ganjar_pranowo", "tipe": "Politikus", "nama": "Ganjar Pranowo"},
    {"id": "hadi_tjahjanto", "tipe": "Politikus", "nama": "Hadi Tjahjanto"},
    {"id": "pdip", "tipe": "Partai", "nama": "PDIP"},
    {"id": "presiden_ri", "tipe": "Jabatan", "nama": "Presiden Republik Indonesia"},
    {"id": "menko_polhukam", "tipe": "Jabatan", "na...
Entitas ditemukan (9):
   [Politikus   ] Joko Widodo
   [Politikus   ] Mahfud MD
   [Politikus   ] Ganjar Pranowo
   [Politikus   ] Hadi Tjahjanto
   [Partai      ] PDIP
   [Jabatan     ] Presiden Republik Indonesia
   [Jabatan     ] Menteri Koordinator Bidang Politik Hukum dan Keam

## Cell 7 - Komponen 3: RAG (Graph-Augmented Retrieval)

**Graph RAG Pipeline:**
1. Terima pertanyaan user dalam bahasa alami
2. Text-to-Cypher -> ambil konteks relevan dari Neo4j (graph traversal)
3. Augment prompt dengan konteks graf
4. LLM generate jawaban berbasis data graf yang akurat

Jawaban LLM **grounded** pada data nyata, bukan halusinasi.


In [ ]:
RAG_SYSTEM = (
    "Anda adalah analis politik Indonesia yang ahli. "
    "Anda memiliki akses ke knowledge graph jaringan politik Indonesia yang berisi data "
    "52.000+ politikus, 156 partai, dan 881 institusi pendidikan beserta relasi antar mereka.\n\n"
    "Gunakan KONTEKS GRAF yang diberikan untuk menjawab pertanyaan dengan akurat dan spesifik. "
    "Jika konteks tidak cukup, sebutkan keterbatasannya. "
    "Jawab dalam Bahasa Indonesia yang informatif dan mudah dipahami. "
    "Sertakan angka dan nama spesifik dari konteks bila tersedia."
)

def retrieve_graph_context(question):
    """Ambil konteks relevan dari graph berdasarkan pertanyaan."""
    contexts = []
    queries_run = []

    # Query 1: Direct Text-to-Cypher
    cypher1 = text_to_cypher(question)
    try:
        results1 = run_query(cypher1)
        if results1:
            contexts.append({
                "sumber": "Text-to-Cypher Direct Query",
                "query": cypher1,
                "data": results1[:15]
            })
        queries_run.append(cypher1)
    except:
        pass

    # Query 2: Statistik umum
    try:
        stats = run_query(
            "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS jumlah ORDER BY jumlah DESC"
        )
        contexts.append({"sumber": "Statistik Database", "data": stats})
    except:
        pass

    # Query 3: Top influencer
    try:
        top_pol = run_query(
            "MATCH (p:Politikus) WHERE p.pagerank IS NOT NULL AND p.nama <> '' "
            "RETURN p.nama AS nama, p.pagerank AS pagerank, p.community AS komunitas "
            "ORDER BY pagerank DESC LIMIT 5"
        )
        if top_pol:
            contexts.append({"sumber": "Top Politikus Berpengaruh", "data": top_pol})
    except:
        pass

    # Query 4: Jika tentang partai
    if any(kw in question.lower() for kw in ["partai", "anggota", "golkar", "pdip", "gerindra", "demokrat"]):
        try:
            partai_data = run_query(
                "MATCH (pol:Politikus)-[:ANGGOTA_PARTAI]->(par:Partai) "
                "WHERE par.nama IS NOT NULL "
                "RETURN par.nama AS partai, count(pol) AS anggota "
                "ORDER BY anggota DESC LIMIT 10"
            )
            if partai_data:
                contexts.append({"sumber": "Data Keanggotaan Partai", "data": partai_data})
        except:
            pass

    # Query 5: Jika tentang pendidikan
    if any(kw in question.lower() for kw in ["universitas", "kampus", "pendidikan", "alumni", "ui ", "ugm", "itb"]):
        try:
            univ_data = run_query(
                "MATCH (p:Politikus)-[:ALUMNI]->(u:Pendidikan) "
                "WHERE u.nama IS NOT NULL AND u.nama <> '' "
                "RETURN u.nama AS universitas, count(p) AS alumni "
                "ORDER BY alumni DESC LIMIT 10"
            )
            if univ_data:
                contexts.append({"sumber": "Data Alumni Politikus per Universitas", "data": univ_data})
        except:
            pass

    # Query 6: Jika tentang dinasti
    if any(kw in question.lower() for kw in ["kerabat", "keluarga", "dinasti", "anak", "cucu", "menantu"]):
        try:
            dinasti = run_query(
                "MATCH (p:Politikus)-[:KERABAT]->(k:Politikus) WHERE p.nama <> '' "
                "RETURN p.nama AS politikus, count(k) AS kerabat_politikus "
                "ORDER BY kerabat_politikus DESC LIMIT 10"
            )
            if dinasti:
                contexts.append({"sumber": "Jaringan Dinasti Politik", "data": dinasti})
        except:
            pass

    return contexts, queries_run

def format_context_for_llm(contexts):
    """Format konteks graf untuk dimasukkan ke prompt LLM."""
    lines = ["=== KONTEKS DARI KNOWLEDGE GRAPH POLITIKUS INDONESIA ===\n"]
    for ctx in contexts:
        lines.append(f"[{ctx['sumber']}]")
        if "query" in ctx:
            lines.append(f"Query: {ctx['query']}")
        for row in ctx.get("data", [])[:10]:
            lines.append(f"  {row}")
        lines.append("")
    return "\n".join(lines)

def rag_query(question):
    """Full RAG Pipeline: Question -> Graph Context -> Augmented LLM Answer."""
    print(f"\n{'='*65}")
    print(f"PERTANYAAN: {question}")
    print("="*65)

    # Step 1: Retrieve
    print("\nStep 1: Mengambil konteks dari knowledge graph...")
    contexts, queries = retrieve_graph_context(question)
    print(f"   {len(contexts)} konteks ditemukan dari {len(queries)} query")

    # Step 2: Augment
    graph_context = format_context_for_llm(contexts)
    augmented_prompt = (
        graph_context +
        f"\n\nPERTANYAAN USER: {question}\n\n"
        "Jawab berdasarkan konteks di atas:"
    )

    # Step 3: Generate
    print("\nStep 2: LLM menganalisis konteks graf...")
    try:
        answer = call_llm(RAG_SYSTEM, augmented_prompt, temperature=0.3)
    except KeyError as e:
        print(f"ERROR LLM: Respon OpenRouter API tidak memiliki kunci 'choices' saat menghasilkan jawaban RAG. Ini mungkin indikasi API key salah, rate limit, atau masalah internal LLM. Pertanyaan: '{question}'. Error: {e}")
        answer = "Maaf, terjadi masalah saat menghasilkan jawaban dengan LLM. Silakan coba lagi nanti atau periksa pengaturan API Anda."
    except Exception as e:
        print(f"ERROR LLM: Terjadi kesalahan tak terduga saat memanggil OpenRouter API untuk pertanyaan RAG: '{question}'. Error: {e}")
        answer = "Maaf, terjadi kesalahan tak terduga saat menghasilkan jawaban dengan LLM. Silakan coba lagi nanti."

    print(f"\n{'-'*65}")
    print("JAWABAN (Graph-Augmented):")
    print(f"{'-'*65}")
    print(answer)

    return answer

# Demo RAG
print("DEMO RAG - Graph-Augmented Retrieval")

rag_questions = [
    "Siapa politikus paling berpengaruh dalam jaringan politik Indonesia berdasarkan data graf?",
    "Bagaimana pola dinasti politik di Indonesia? Siapa keluarga yang paling dominan?",
    "Universitas mana yang paling banyak mencetak politikus dan apa implikasinya?",
]

rag_answers = {}
for q in rag_questions:
    answer = rag_query(q)
    rag_answers[q] = answer
    print()


DEMO RAG - Graph-Augmented Retrieval

PERTANYAAN: Siapa politikus paling berpengaruh dalam jaringan politik Indonesia berdasarkan data graf?

Step 1: Mengambil konteks dari knowledge graph...
   3 konteks ditemukan dari 1 query

Step 2: LLM menganalisis konteks graf...

-----------------------------------------------------------------
JAWABAN (Graph-Augmented):
-----------------------------------------------------------------
Berdasarkan data **PageRank** dari knowledge graph jaringan politik Indonesia, politikus paling berpengaruh adalah:

## **Zakaria bin Muhammad Amin**
- **Skor PageRank: 11.11** (tertinggi di seluruh jaringan)
- **Komunitas/Cluster: 52.583** (terhubung dalam komunitas terbesar)

---

### **10 Politikus Teratas Berdasarkan PageRank:**

| Peringkat | Nama | Skor PageRank |
|-----------|------|---------------|
| 1 | **Zakaria bin Muhammad Amin** | **11.11** |
| 2 | Sukarno | 9.13 |
| 3 | Basuki Tjahaja Purnama (Ahok) | 6.46 |
| 4 | Megawati Soekarnoputri | 5.92 |
| 5 

## Cell 8 - Interactive Political Graph Chatbot

Gabungkan semua komponen dalam antarmuka chatbot interaktif.
Ketik pertanyaan tentang jaringan politik Indonesia.


In [ ]:
def political_chatbot(question, mode="rag"):
    """
    Chatbot graf politik Indonesia.

    mode: 'rag'    = full RAG pipeline (direkomendasikan)
          'cypher' = text-to-cypher + ringkasan LLM
          'direct' = LLM tanpa graph context
    """
    if mode == "rag":
        return rag_query(question)
    elif mode == "cypher":
        results = ask_graph(question)
        if results:
            ctx = "\n".join([str(r) for r in results[:10]])
            return call_llm(
                "Anda adalah analis politik Indonesia. Rangkum data berikut dalam kalimat informatif.",
                f"Pertanyaan: {question}\nData: {ctx}"
            )
        return "Tidak ada data ditemukan."
    else:
        return call_llm(
            "Anda adalah pakar politik Indonesia.",
            question,
            temperature=0.5
        )

# Demo chatbot
print("POLITICAL GRAPH CHATBOT - Siap digunakan!")
print()
print("Cara penggunaan:")
print('  political_chatbot("Siapa politikus dengan jaringan kerabat terluas?")')
print('  political_chatbot("Partai apa yang paling dominan?", mode="rag")')
print()

# Demo satu pertanyaan
demo_answer = political_chatbot(
    "Berikan analisis singkat tentang distribusi keanggotaan partai berdasarkan data graf."
)


POLITICAL GRAPH CHATBOT - Siap digunakan!

Cara penggunaan:
  political_chatbot("Siapa politikus dengan jaringan kerabat terluas?")
  political_chatbot("Partai apa yang paling dominan?", mode="rag")


PERTANYAAN: Berikan analisis singkat tentang distribusi keanggotaan partai berdasarkan data graf.

Step 1: Mengambil konteks dari knowledge graph...
   4 konteks ditemukan dari 1 query

Step 2: LLM menganalisis konteks graf...

-----------------------------------------------------------------
JAWABAN (Graph-Augmented):
-----------------------------------------------------------------
## Analisis Distribusi Keanggotaan Partai (Berdasarkan Knowledge Graph)

### **Gambaran Umum**
- **Total politikus**: 52.301 orang
- **Total partai tercatat**: 159 partai
- **Top 10 partai** menguasai **~64,15%** total keanggotaan (33.576 politikus)

### **Pola Distribusi: Relatif Merata di Atas**
| Peringkat | Partai | Anggota | Persentase |
|-----------|--------|---------|------------|
| 1 | **Golkar** | 3.

## Cell 9 - Evaluasi & Ringkasan Sistem

Evaluasi kualitas setiap komponen dan ringkasan keseluruhan sistem.


In [ ]:
import time

print("=" * 65)
print("EVALUASI SISTEM - Graf Politikus Indonesia Tier 4")
print("=" * 65)

# 1. Evaluasi Text-to-Cypher
print("\n[1] TEXT-TO-CYPHER EVALUATION")
print("-" * 40)
test_cases = [
    "Berapa total politikus dalam database?",
    "Partai dengan anggota terbanyak?",
    "Siapa 3 politikus pagerank tertinggi?",
]
t2c_success = 0
for q in test_cases:
    t0 = time.time()
    cypher = text_to_cypher(q)
    try:
        result = run_query(cypher)
        status = "PASS" if result else "EMPTY"
        t2c_success += 1 if result else 0
    except Exception as e:
        status = f"FAIL ({str(e)[:30]})"
    elapsed = time.time() - t0
    print(f"  {status:6s} | {q[:50]:50s} | {elapsed:.2f}s")

print(f"\n  Akurasi Text-to-Cypher: {t2c_success}/{len(test_cases)} ({100*t2c_success//len(test_cases)}%)")

# 2. Evaluasi Graph Builder
print("\n[2] GRAPH BUILDER EVALUATION")
print("-" * 40)
be_stats = run_query(
    "MATCH (n:BeritaEntity) "
    "RETURN count(n) AS total, count(DISTINCT n.tipe) AS tipe_unik"
)
if be_stats:
    print(f"  Total BeritaEntity : {be_stats[0]['total']}")
    print(f"  Tipe entitas unik  : {be_stats[0]['tipe_unik']}")

ref_count = run_query("MATCH (be:BeritaEntity)-[:REFERS_TO]->() RETURN count(be) AS c")[0]["c"]
print(f"  Entitas ter-link ke graph utama: {ref_count}")

# 3. Evaluasi Graph Analytics
print("\n[3] GRAPH ANALYTICS EVALUATION")
print("-" * 40)
for prop, label, name in [
    ("pagerank",    "Politikus", "PageRank"),
    ("betweenness", "Politikus", "Betweenness Centrality"),
    ("community",   "Politikus", "Louvain Community"),
]:
    try:
        count = run_query(
            f"MATCH (n:{label}) WHERE n.{prop} IS NOT NULL RETURN count(n) AS c"
        )[0]["c"]
        status = "OK" if count > 0 else "EMPTY"
        print(f"  {status:6s} {name}: {count:,} nodes")
    except:
        print(f"  FAIL  {name}: tidak tersedia")

# 4. Evaluasi RAG
print("\n[4] RAG PIPELINE EVALUATION")
print("-" * 40)
t0 = time.time()
contexts, _ = retrieve_graph_context("Siapa politikus paling berpengaruh?")
rag_elapsed = time.time() - t0
total_rows = sum(len(c.get("data", [])) for c in contexts)
print(f"  Context retrieval : {len(contexts)} sumber ({rag_elapsed:.2f}s)")
print(f"  Total data rows   : {total_rows}")
print(f"  RAG augmentation  : aktif")

# 5. Database Summary
print("\n[5] DATABASE SUMMARY")
print("-" * 40)
for r in run_query("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS jumlah ORDER BY jumlah DESC"):
    print(f"  {r['label']:20s}: {r['jumlah']:>8,} nodes")
print()
for r in run_query("MATCH ()-[r]->() RETURN type(r) AS relasi, count(r) AS jumlah"):
    print(f"  {r['relasi']:25s}: {r['jumlah']:>8,} relasi")

# Tier 4 Checklist
print("\n" + "=" * 65)
print("TIER 4 CHECKLIST")
print("=" * 65)
for status, komponen, keterangan in [
    ("OK", "LLM Text-to-Cypher", "NL -> Cypher query yang valid"),
    ("OK", "LLM Graph Builder",  "Ekstraksi entitas teks -> Neo4j nodes/edges"),
    ("OK", "Graph Analytics",    "PageRank, Betweenness, Louvain via GDS"),
    ("OK", "RAG Pipeline",       "Graph-Augmented Retrieval jawaban faktual"),
    ("OK", "Neo4j AuraDB",       "Cloud database dengan 50K+ nodes"),
    ("OK", "OpenRouter API",     "Free LLM model via API"),
    ("OK", "Multi-entitas",      "Politikus + Partai + Pendidikan + Person"),
    ("OK", ">50 nodes",          "52.295+ Politikus saja"),
]:
    print(f"  [{status}] {komponen:25s} - {keterangan}")

print("\n" + "="*65)
print("Estimasi Nilai: Tier 4 -> 90-100")
print("="*65)
print("\nNotebook selesai! Gunakan political_chatbot() untuk query interaktif.")


EVALUASI SISTEM - Graf Politikus Indonesia Tier 4

[1] TEXT-TO-CYPHER EVALUATION
----------------------------------------
  PASS   | Berapa total politikus dalam database?             | 15.17s
ERROR LLM: Terjadi kesalahan tak terduga saat memanggil OpenRouter API untuk pertanyaan: 'Partai dengan anggota terbanyak?'. Error: Expecting value: line 425 column 1 (char 2332)
  FAIL (Cannot run an empty query) | Partai dengan anggota terbanyak?                   | 89.32s
  PASS   | Siapa 3 politikus pagerank tertinggi?              | 17.72s

  Akurasi Text-to-Cypher: 2/3 (66%)

[2] GRAPH BUILDER EVALUATION
----------------------------------------
  Total BeritaEntity : 13
  Tipe entitas unik  : 3
  Entitas ter-link ke graph utama: 14

[3] GRAPH ANALYTICS EVALUATION
----------------------------------------
  OK     PageRank: 52,295 nodes
  OK     Betweenness Centrality: 52,295 nodes
  OK     Louvain Community: 52,295 nodes

[4] RAG PIPELINE EVALUATION
----------------------------------------
 

## Cell 10 - Custom Query (Opsional)

In [ ]:
# Custom Query - Tulis query Anda sendiri di sini
# =====================================================

# Contoh 1: Text-to-Cypher
# ask_graph("Siapa politikus yang menjadi anggota lebih dari satu partai?")

# Contoh 2: RAG Chatbot
# political_chatbot(
#     "Bagaimana hubungan antara latar belakang pendidikan dan afiliasi partai?"
# )

# Contoh 3: Graph Builder dari teks baru
# berita_baru = "...tulis teks berita politik di sini..."
# extracted = extract_entities_from_text(berita_baru)
# stats = populate_graph_from_extraction(extracted, berita_baru)
# print(stats)

# Contoh 4: Direct Cypher Query
# results = run_query(
#     "MATCH (p:Politikus)-[:ANGGOTA_PARTAI]->(par:Partai) "
#     "WHERE toLower(par.nama) CONTAINS 'demokrat' "
#     "RETURN p.nama, par.nama LIMIT 20"
# )
# for r in results:
#     print(r)

print("Sel ini siap untuk custom query Anda.")
print("Uncomment salah satu contoh di atas atau tulis query baru.")
